## Calibration

In [ ]:
from pynq.lib import MicroblazeLibrary
from pynq.overlays.base import BaseOverlay
base = BaseOverlay("base.bit")
lib = MicroblazeLibrary(base.PMODB, ['i2c'])
from time import sleep
import math

## constant.py
TCA_DEVICE_ADDRESS = 0x70
TCA_MOTOR_CHANNEL = 0
TCA_MPU_CHANNEL = 1


# PCA9685
PCA9685_ADDRESS = 0x40
PCA9685_MODE1 = 0x00
PCA9685_MODE2 = 0x01
SUBADR1 = 0x02
SUBADR2 = 0x03
SUBADR3 = 0x04
PCA9685_PRESCALE = 0xFE
LED0_ON_L = 0x06
LED0_ON_H = 0x07
LED0_OFF_L = 0x08
LED0_OFF_H = 0x09
PCA9685_RESTART = 0x80
PCA9685_SLEEP = 0x10
PCA9685_EXTCLK = 0x40
PCA9685_ALLCALL = 0x01
PCA9685_INVRT = 0x10
PCA9685_OUTDRV = 0x04

## Pmod I2C Pin Definitions
PMOD_SDA_PIN = 6  ## or 2
PMOD_SCL_PIN = 7  ## or 3


# Motor
MIN_MOTOR_FREQ_HZ = 1000
MAX_MOTOR_FREQ_HZ = 2000


## pynq_i2c.py
class pynq_i2c:
    def __init__(self):
        self.i2c_device = lib.i2c_open(PMOD_SDA_PIN, PMOD_SCL_PIN)
    
    def iic_writeByte(self, devAddr, regAddr, data):
        temp = bytearray(2)
        temp[0] = regAddr
        temp[1] = data
        self.i2c_device.write(devAddr, temp, 2)
    
    
    def iic_readByte(self, devAddr, regAddr):
        temp = bytearray(1)
        temp[0] = regAddr
        self.i2c_device.write(devAddr, temp, 1)
        self.i2c_device.read(devAddr, temp, 1)    
        return temp[0]
    
    def close(self):
        self.i2c_device.close()
        
pynq_i2c_instance = pynq_i2c()


## utils.py
# Convert microseconds to PWM range used by PCA9685
# Assume a 20ms period
def us_to_pwm(value_us: int) -> int:
    return int(value_us * 4096 / 20000)


## PCA.py
# https://github.com/adafruit/Adafruit_Python_PCA9685/blob/master/Adafruit_PCA9685/PCA9685.py
def PCA_init():    
    pynq_i2c_instance.iic_writeByte(PCA9685_ADDRESS, PCA9685_MODE2, PCA9685_OUTDRV)
    pynq_i2c_instance.iic_writeByte(PCA9685_ADDRESS, PCA9685_MODE1, PCA9685_ALLCALL)
    sleep(0.005)
    mode1 = pynq_i2c_instance.iic_readByte(PCA9685_ADDRESS, PCA9685_MODE1)
    mode1 = mode1 & ~PCA9685_SLEEP 
    pynq_i2c_instance.iic_writeByte(PCA9685_ADDRESS, PCA9685_MODE1, mode1)
    sleep(0.005)
    
    set_pwm_freq(1000)

    
def set_pwm_freq(freq_hz):
    prescaleval = 25000000.0    # 25MHz
    prescaleval /= 4096.0       # 12-bit
    prescaleval /= float(freq_hz)
    prescaleval -= 1.0
    prescale = int(math.floor(prescaleval + 0.5))
    oldmode = pynq_i2c_instance.iic_readByte(PCA9685_ADDRESS, PCA9685_MODE1)
    newmode = (oldmode & 0x7F) | 0x10    # sleep
    pynq_i2c_instance.iic_writeByte(PCA9685_ADDRESS, PCA9685_MODE1, newmode)  # go to sleep
    pynq_i2c_instance.iic_writeByte(PCA9685_ADDRESS, PCA9685_PRESCALE, prescale)
    pynq_i2c_instance.iic_writeByte(PCA9685_ADDRESS, PCA9685_MODE1, oldmode)
    sleep(0.005)
    pynq_i2c_instance.iic_writeByte(PCA9685_ADDRESS, PCA9685_MODE1, oldmode | 0x80)


def set_pwm(motor, on, off):
    """Sets a single PWM channel."""
    channel = motor_to_channel(motor)
    pynq_i2c_instance.iic_writeByte(PCA9685_ADDRESS, LED0_ON_L+4*channel, on & 0xFF)
    pynq_i2c_instance.iic_writeByte(PCA9685_ADDRESS, LED0_ON_H+4*channel, on >> 8)
    pynq_i2c_instance.iic_writeByte(PCA9685_ADDRESS, LED0_OFF_L+4*channel, off & 0xFF)
    pynq_i2c_instance.iic_writeByte(PCA9685_ADDRESS, LED0_OFF_H+4*channel, off >> 8)

def motor_to_channel(motor) -> int:
    match motor:
        case 1:
                return 3
        case 2:
                return 4
        case 3:
                return 7
        case 4:
                return 8

### TCA.py
def channel_select(channel):
    """Select an individual channel."""
    if channel > 7:
        return
    pynq_i2c_instance.iic_writeByte(TCA_DEVICE_ADDRESS, 0x00, 1 << channel)
    
    



In [ ]:
channel_select(TCA_MOTOR_CHANNEL)
PCA_init()
print("PCA9685 initialized")

In [ ]:
def calibrate_esc(motor):
    min_pwm = us_to_pwm(MIN_MOTOR_FREQ_HZ)
    max_pwm = us_to_pwm(MAX_MOTOR_FREQ_HZ-100)

#     input("Send Min...")
#     set_pwm(motor, 0, min_pwm)
#     print("Wait for 1 Bip\n")

    input("Send Max...")
    set_pwm(motor, 0, max_pwm)
    print("Wait for 4 Bip, followed by 3 RAMP UP\n")
    
#     input("Send Max...")
#     set_pwm(motor, 0, max_pwm)
#     print("Wait for 4 Bip, followed by 3 RAMP UP\n")

    input(f"Send Min...")
    set_pwm(motor, 0, min_pwm)
    print("Wait for 4 double Bip, followed by 3 RAMP DOWN\n")

    print(f"Calibration complete for Motor {motor}!\n")


In [ ]:
motor_to_calibrate = 1
calibrate_esc(motor_to_calibrate)

Send Max...
Wait for 4 Bip, followed by 3 RAMP UP



In [25]:
set_pwm(motor_to_calibrate, 0, us_to_pwm(1100)) 

In [26]:
set_pwm(motor_to_calibrate, 0, us_to_pwm(0)) 

## Testing

In [28]:
from pynq.overlays.base import BaseOverlay
from pynq.lib import MicroblazeLibrary
from pynq import Overlay

base = BaseOverlay("base.bit")
# lib = MicroblazeLibrary(base.RPI, ['i2c'])
lib = MicroblazeLibrary(base.PMODB, ['i2c'])

In [35]:
from time import sleep
import math

# TCA9548A
TCA_DEVICE_ADDRESS = 0x70


## Pmod I2C Pin Definitions
PMOD_SDA_PIN = 6  ## or 2
PMOD_SCL_PIN = 7  ## or 3
# RPI_SDA_PIN = 3
# RPI_SCL_PIN = 5


# PCA9685
PCA9685_ADDRESS = 0x40
PCA9685_MODE1 = 0x00
PCA9685_MODE2 = 0x01
SUBADR1 = 0x02
SUBADR2 = 0x03
SUBADR3 = 0x04
PCA9685_PRESCALE = 0xFE
LED0_ON_L = 0x06
LED0_ON_H = 0x07
LED0_OFF_L = 0x08
LED0_OFF_H = 0x09

# Bits:
PCA9685_RESTART = 0x80
PCA9685_SLEEP = 0x10
PCA9685_EXTCLK = 0x40
PCA9685_ALLCALL = 0x01
PCA9685_INVRT = 0x10
PCA9685_OUTDRV = 0x04


# i2c_device = lib.i2c_open(RPI_SDA_PIN, RPI_SCL_PIN)
i2c_device = lib.i2c_open(PMOD_SDA_PIN, PMOD_SCL_PIN)

def iic_writeByte(devAddr, regAddr, data):
    temp = bytearray(2)
    temp[0] = regAddr
    temp[1] = data
    i2c_device.write(devAddr, temp, 2)
    
    
def iic_readByte(devAddr, regAddr):
    temp = bytearray(1)
    temp[0] = regAddr
    i2c_device.write(devAddr, temp, 1)
    i2c_device.read(devAddr, temp, 1)    
    return temp[0]


def TCA_channel_select(channel):
    """Select an individual channel."""
    if channel > 7:
        return
    iic_writeByte(TCA_DEVICE_ADDRESS, 0x00, 1 << channel)
    sleep(1)

# https://github.com/adafruit/Adafruit_Python_PCA9685/blob/master/Adafruit_PCA9685/PCA9685.py
def PCA_Init():    
    iic_writeByte(PCA9685_ADDRESS, PCA9685_MODE2, PCA9685_OUTDRV)
    iic_writeByte(PCA9685_ADDRESS, PCA9685_MODE1, PCA9685_ALLCALL)
    sleep(0.005)
    mode1 = iic_readByte(PCA9685_ADDRESS, PCA9685_MODE1)
    mode1 = mode1 & ~PCA9685_SLEEP 
    iic_writeByte(PCA9685_ADDRESS, PCA9685_MODE1, mode1)
    sleep(0.005)
    
    set_pwm_freq(1000)


def set_pwm_freq(freq_hz):
    prescaleval = 25000000.0    # 25MHz
    prescaleval /= 4096.0       # 12-bit
    prescaleval /= float(freq_hz)
    prescaleval -= 1.0
    prescale = int(math.floor(prescaleval + 0.5))
    oldmode = iic_readByte(PCA9685_ADDRESS, PCA9685_MODE1)
    newmode = (oldmode & 0x7F) | 0x10    # sleep
    iic_writeByte(PCA9685_ADDRESS, PCA9685_MODE1, newmode)  # go to sleep
    iic_writeByte(PCA9685_ADDRESS, PCA9685_PRESCALE, prescale)
    iic_writeByte(PCA9685_ADDRESS, PCA9685_MODE1, oldmode)
    sleep(0.005)
    iic_writeByte(PCA9685_ADDRESS, PCA9685_MODE1, oldmode | 0x80)
    
    

def set_pwm(channel, on, off):
    """Sets a single PWM channel."""
    iic_writeByte(PCA9685_ADDRESS, LED0_ON_L+4*channel, on & 0xFF)
    iic_writeByte(PCA9685_ADDRESS, LED0_ON_H+4*channel, on >> 8)
    iic_writeByte(PCA9685_ADDRESS, LED0_OFF_L+4*channel, off & 0xFF)
    iic_writeByte(PCA9685_ADDRESS, LED0_OFF_H+4*channel, off >> 8)

    
def arm_motors():
    channels = [3, 4, 7, 8]
    
     # Convert microseconds to PWM range used by PCA9685
    def us_to_pwm(value_us):
        return int(value_us * 4096 / 20000)  # 20ms period
    
    min_pwm = us_to_pwm(1000)
    max_pwm = us_to_pwm(2000)
    
    for channel in channels:
        sleep(0.5)
        set_pwm(channel, 0, min_pwm)
        sleep(1)
        set_pwm(channel, 0, max_pwm)
        sleep(1)
        set_pwm(channel, 0, min_pwm)
        sleep(0.5)
        
    print("ARMING SEQUENCE DONE")

    
def start_motors(speed_us=1000):
    """
    Start the motors with the specified speed.
    
    Parameters:
    speed_us (int): Desired pulse width in microseconds for motor speed (typically between 1100µs and 1900µs).
    """
    channels = [3, 4, 7, 8]
    
    # Convert microseconds to PWM range used by PCA9685
    def us_to_pwm(value_us):
        return int(value_us * 4096 / 20000)  # 20ms period
    
    speed_pwm = us_to_pwm(speed_us)
    
    for channel in channels:
        set_pwm(channel, 0, speed_pwm)
    
    print("MOTORS STARTED WITH SPEED PWM VALUE: ", speed_pwm)
    

print("TCA SELECT CHANNEL 0")
TCA_channel_select(0)

print("INITIALIZE PWM DRIVER")
PCA_Init()

TCA SELECT CHANNEL 0
INITIALIZE PWM DRIVER


In [36]:
print("STARTING ARMING SEQUENCE")
arm_motors()

STARTING ARMING SEQUENCE
ARMING SEQUENCE DONE


In [37]:
start_motors(1500)

MOTORS STARTED WITH SPEED PWM VALUE:  307


In [ ]:
start_motors(0)